# Day 098 | Documentation & README Polish

## 100 Days of ML for Geotechnical Engineering
**Phase:** Geotechnical AI Projects | **Author:** Ripon Chandra Malo | **Date:** 2026-05-16

---

## Why This Day Matters

You have trained models. You have engineered features. You have evaluated performance.  
But **none of it matters** if your collaborator — or your future self — cannot reproduce, trust, or use your work.

> *"Code without documentation is like a borehole without a log: you drove the rod, but nobody knows what you found."*

### What We Cover Today

| Part | Topic |
|------|-------|
| 1 | Why documentation matters for engineering ML |
| 2 | Google-style docstrings — bad vs good |
| 3 | Type hints for function signatures |
| 4 | Complete model card for liquefaction prediction |
| 5 | Experiment tracking template |
| 6 | Reproducibility checklist |
| 7 | How to write a geotechnical ML README |
| 8 | Fully documented end-to-end function demo |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import json
import hashlib
import datetime
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

print('Imports OK.')

---

## Part 1 — Why Documentation Matters for Engineering ML

Engineering ML is different from academic ML. Your model's output may inform:

- Pile design loads worth millions of dollars
- Slope stability decisions affecting public safety
- Ground improvement specifications on critical infrastructure

### The Cost of Poor Documentation

| Scenario | Without docs | With docs |
|----------|-------------|----------|
| New engineer joins | Weeks to understand model | Hours — README explains everything |
| Regulator asks for audit | Cannot reproduce results | Dataset version + seed → exact replication |
| Model fails in production | Unknown which data was used | Experiment log traces every run |
| Client questions a prediction | "The model said so" | Model card states limitations clearly |
| You return after 6 months | What did this function do? | Docstring explains assumptions and units |

### Documentation Hierarchy for an Engineering ML Project

```
Level 1  │  Function docstrings    — what, args, returns, raises, examples
Level 2  │  Module-level comments  — purpose, author, date, references
Level 3  │  Experiment logs        — hyperparams, metrics, decisions
Level 4  │  Model card             — intended use, limitations, fairness
Level 5  │  README                 — project overview, installation, usage
```

---

## Part 2 — Google-Style Docstrings: Bad vs Good

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# BAD DOCSTRING — What NOT to do
# ─────────────────────────────────────────────────────────────────────────

def liquefaction_CRR(N60, sigma_v_prime, fines_content=0):
    """Calculate CRR."""
    Ncs = N60 + (0.5 + 3.6 / (fines_content + 0.1)) if fines_content > 5 else N60
    CRR75 = np.exp(Ncs/18.9 - (Ncs/29.5)**2 - (Ncs/13.3)**4 + 45.9/126.1 - 2.8)
    return CRR75

# Problems with this docstring:
# 1. Says nothing about what CRR means
# 2. No units stated for any parameter
# 3. No mention of what standard/paper the formula comes from
# 4. No return description
# 5. Mysterious magic numbers — 18.9, 29.5, 13.3, 45.9, 126.1, 2.8 — unexplained
print('Bad docstring example defined.')
print('Problem: nobody knows what the function does, what units to pass, or what formula it uses.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# GOOD DOCSTRING — Google-style
# ─────────────────────────────────────────────────────────────────────────

def liquefaction_CRR_M75(
    N60: float,
    sigma_v_prime: float,
    fines_content: float = 0.0,
) -> float:
    """
    Cyclic Resistance Ratio (CRR) for M7.5 earthquakes from SPT data.

    Implements the Idriss & Boulanger (2004) relationship for clean sand
    equivalent normalised blow count (Ncs), using the polynomial
    approximation to the CRR–Ncs boundary curve.

    Reference:
        Idriss, I.M. & Boulanger, R.W. (2004). Semi-empirical procedures
        for evaluating liquefaction potential during earthquakes.
        Proc. 11th ICSDEE, Vol. 1, pp. 32–56.

    Args:
        N60: SPT blow count corrected for energy ratio (60% efficiency),
            overburden stress, borehole diameter, rod length, and sampler
            type. Units: blows/300 mm. Range: [0, 60].
        sigma_v_prime: Effective vertical stress at the depth of interest.
            Units: kPa. Must be > 0.
        fines_content: Percentage of fines (particles < 0.075 mm) by mass.
            Units: %. Default 0 (clean sand). Range: [0, 100].

    Returns:
        CRR75: Cyclic resistance ratio calibrated for M7.5 earthquakes.
            Dimensionless. Typical range: [0.05, 0.60].
            Returns None if N60 > 30 (dense soil, no liquefaction expected).

    Raises:
        ValueError: If sigma_v_prime <= 0.
        ValueError: If fines_content is outside [0, 100].

    Example:
        >>> crr = liquefaction_CRR_M75(N60=14, sigma_v_prime=80.0, fines_content=5.0)
        >>> print(f'CRR (M7.5) = {crr:.3f}')  # typical: ~0.15
    """
    if sigma_v_prime <= 0:
        raise ValueError(f'sigma_v_prime must be positive, got {sigma_v_prime}')
    if not (0 <= fines_content <= 100):
        raise ValueError(f'fines_content must be in [0, 100], got {fines_content}')

    # Dense soil — no liquefaction expected
    if N60 > 30:
        return None

    # Fines correction: convert to clean-sand equivalent (Ncs)
    if fines_content > 5:
        delta_N = 0.5 + 3.6 / (fines_content + 0.1)   # Idriss & Boulanger (2004) Eq. 2.15
        Ncs = N60 + delta_N
    else:
        Ncs = float(N60)

    # Polynomial approximation to the CRR boundary curve — Eq. 2.24
    CRR75 = np.exp(
        Ncs / 18.9
        - (Ncs / 29.5) ** 2
        - (Ncs / 13.3) ** 4
        + 45.9 / 126.1
        - 2.8
    )
    return round(CRR75, 4)


# Demo: typical site conditions
print('CRR demonstrations:')
test_cases = [
    (8,  50.0,  0, 'Loose clean sand, shallow'),
    (14, 80.0,  5, 'Medium-dense sand, moderate depth'),
    (22, 120.0, 15, 'Dense silty sand, deep'),
    (35, 150.0,  0, 'Very dense sand (no liquefaction)'),
]
print(f'{"Description":<42} {"N60":>4} {"CRR":>8}')
print('-' * 58)
for N60, sv, fc, desc in test_cases:
    crr = liquefaction_CRR_M75(N60, sv, fc)
    crr_str = f'{crr:.4f}' if crr is not None else 'N/A (dense)'
    print(f'{desc:<42} {N60:>4d} {crr_str:>8}')

---

## Part 3 — Type Hints for Function Signatures

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TYPE HINTS — Before and after
# ─────────────────────────────────────────────────────────────────────────
from typing import Union

# WITHOUT type hints — ambiguous
def predict_settlement(E, H, q, nu=0.3):
    """Elastic settlement under a uniformly loaded strip footing."""
    return (q * H * (1 - nu**2)) / E

# WITH type hints — self-documenting
def predict_settlement_typed(
    E: float,             # Young's modulus (MPa)
    H: float,             # Layer thickness (m)
    q: float,             # Bearing pressure (kPa)
    nu: float = 0.3,      # Poisson's ratio (dimensionless)
) -> float:
    """Elastic settlement of a compressible layer under uniform load.

    Uses the simplified Boussinesq elastic formula.

    Args:
        E:  Drained Young's modulus in MPa.  Typical: 5–50 for clay, 20–80 for sand.
        H:  Thickness of the compressible layer in metres.
        q:  Net applied bearing pressure in kPa.
        nu: Drained Poisson's ratio (dimensionless). Default 0.30.

    Returns:
        Elastic settlement in millimetres.
    """
    E_kPa = E * 1000.0           # convert MPa → kPa for consistent units
    settlement_m = (q * H * (1.0 - nu**2)) / E_kPa
    return round(settlement_m * 1000.0, 2)   # metres → mm


# Advanced: complex return types
def classify_soil_USCS(
    LL: float,
    PL: float,
    percent_fines: float,
    D60: Optional[float] = None,
    D10: Optional[float] = None,
) -> Tuple[str, str]:
    """Classify soil using the Unified Soil Classification System (USCS).

    Args:
        LL: Liquid limit (%). Required.
        PL: Plastic limit (%). Required.
        percent_fines: Percentage passing 0.075 mm sieve (%).
        D60: Grain diameter at 60% passing (mm). Required for coarse soils.
        D10: Grain diameter at 10% passing (mm). Required for coarse soils.

    Returns:
        Tuple of (symbol, description). Example: ('CH', 'Fat Clay').
    """
    PI = LL - PL

    if percent_fines > 50:   # Fine-grained soil
        if LL >= 50:
            if PI > 0.73 * (LL - 20):
                return ('CH', 'Fat Clay')
            else:
                return ('MH', 'Elastic Silt')
        else:
            if PI > 7 and PI > 0.73 * (LL - 20):
                return ('CL', 'Lean Clay')
            elif PI < 4 or PI < 0.73 * (LL - 20):
                return ('ML', 'Silt')
            else:
                return ('CL-ML', 'Silty Clay')
    else:                    # Coarse-grained (simplified)
        return ('SP', 'Poorly-Graded Sand')  # simplified for demo


print('Settlement predictions (with type hints):')
cases = [(10, 5.0, 100), (25, 3.0, 200), (50, 8.0, 150)]
for E, H, q in cases:
    s = predict_settlement_typed(E, H, q)
    print(f'  E={E:>3} MPa | H={H:.1f}m | q={q} kPa  → Settlement = {s:.1f} mm')

print('\nUSCS classifications:')
soils = [
    (65, 30, 75, None, None, 'High-plasticity clay'),
    (35, 18, 65, None, None, 'Medium-plasticity clay'),
    (25, 12, 60, None, None, 'Low-plasticity silt'),
]
for LL, PL, pf, D60, D10, name in soils:
    symbol, desc = classify_soil_USCS(LL, PL, pf, D60, D10)
    print(f'  {name:<30}  →  {symbol}  ({desc})')

---

## Part 4 — Model Card for a Liquefaction Prediction Model

A **model card** (Mitchell et al., 2019) is a structured document that every deployed ML model should have. It tells stakeholders exactly what the model can and cannot do.

---

### Model Card: `LiquefactionNet-v1.2`

#### Model Details
- **Model type:** Gradient Boosting Classifier (XGBoost 2.0)
- **Version:** 1.2.0 (2026-03-15)
- **Author:** Ripon Chandra Malo
- **Training framework:** scikit-learn 1.4 + XGBoost 2.0

#### Intended Use
- Screening tool for **preliminary** liquefaction susceptibility during site investigation planning
- Prioritising boreholes for detailed cyclic testing (cyclic triaxial, CPTU)
- Educational demonstrations of ML for seismic hazard

#### Out-of-Scope Uses (Do NOT use for)
- Final design or regulatory compliance liquefaction assessment (use Idriss-Boulanger or Youd-Idriss)
- Sites outside the training data distribution (see limitations)
- Areas with organic soils, cemented sands, or gravels > 10% gravel content

#### Training Data
- **Source:** PEER liquefaction database, USGS earthquake case histories, JAEA SPT database
- **Size:** 1 847 case histories (liquefied: 612, non-liquefied: 1 235)
- **Features:** Depth, N60, fines content, WT depth, PGA, Mw, Vs30
- **Time period:** Earthquakes from 1964 (Niigata) to 2016 (Kumamoto)

#### Performance Metrics

| Split | AUC-ROC | Precision | Recall | F1 |
|-------|---------|-----------|--------|----|
| Train (5-fold CV) | 0.923 | 0.87 | 0.82 | 0.84 |
| Test (hold-out)   | 0.901 | 0.83 | 0.79 | 0.81 |

#### Limitations & Known Biases
1. **Geographic bias:** 68% of training data from Japan and California. Performance in South/Southeast Asia, Africa, and South America is unvalidated.
2. **Depth limit:** Trained on depths 1–20 m. Do not extrapolate below 20 m.
3. **Magnitude range:** Training Mw = 5.9–7.9. Predictions outside this range are unreliable.
4. **Class imbalance:** Non-liquefied cases outnumber liquefied 2:1. Recall on liquefied class may be slightly underestimated.
5. **N60 uncertainty:** SPT blow count variability ±20% is not propagated through the model.

---

## Part 5 — Experiment Tracking Template

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# EXPERIMENT TRACKING — Dataclass-based experiment log
# ─────────────────────────────────────────────────────────────────────────

@dataclass
class ExperimentRun:
    """Immutable record of a single ML experiment run."""

    # Identification
    run_id:          str
    timestamp:       str
    author:          str

    # Data
    dataset_name:    str
    dataset_version: str
    dataset_hash:    str                    # SHA-256 of CSV for reproducibility
    n_train:         int
    n_test:          int
    features:        List[str]
    target:          str

    # Model
    model_class:     str
    hyperparameters: Dict
    random_seed:     int

    # Metrics
    cv_r2_mean:      float
    cv_r2_std:       float
    test_r2:         float
    test_mae:        float
    test_rmse:       float

    # Decision
    promoted_to_staging: bool = False
    notes:               str  = ''


def hash_file(filepath: str) -> str:
    """Return SHA-256 hex digest of a file. Used to pin dataset versions."""
    h = hashlib.sha256()
    try:
        with open(filepath, 'rb') as f:
            h.update(f.read())
    except FileNotFoundError:
        return 'file_not_found'
    return h.hexdigest()[:16]   # first 16 hex chars is enough


# Simulate two experiment runs
run_1 = ExperimentRun(
    run_id='EXP-2026-041',
    timestamp=datetime.datetime.now().isoformat(timespec='seconds'),
    author='Ripon Chandra Malo',
    dataset_name='pile_load_tests.csv',
    dataset_version='v1.0',
    dataset_hash=hash_file('pile_load_tests.csv'),
    n_train=160, n_test=40,
    features=['pile_diameter_m', 'pile_length_m', 'avg_N_value',
              'shaft_friction_kPa', 'tip_resistance_kPa'],
    target='load_test_capacity_kN',
    model_class='GradientBoostingRegressor',
    hyperparameters={'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.10},
    random_seed=42,
    cv_r2_mean=0.912, cv_r2_std=0.018,
    test_r2=0.905, test_mae=142.3, test_rmse=218.7,
    promoted_to_staging=False,
    notes='Baseline run with raw features only.'
)

run_2 = ExperimentRun(
    run_id='EXP-2026-042',
    timestamp=datetime.datetime.now().isoformat(timespec='seconds'),
    author='Ripon Chandra Malo',
    dataset_name='pile_load_tests.csv',
    dataset_version='v1.0',
    dataset_hash=hash_file('pile_load_tests.csv'),
    n_train=160, n_test=40,
    features=['pile_diameter_m', 'pile_length_m', 'avg_N_value',
              'shaft_friction_kPa', 'tip_resistance_kPa',
              'aspect_ratio', 'shaft_area_m2', 'shaft_capacity_kN'],
    target='load_test_capacity_kN',
    model_class='GradientBoostingRegressor',
    hyperparameters={'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05},
    random_seed=42,
    cv_r2_mean=0.951, cv_r2_std=0.012,
    test_r2=0.947, test_mae=98.1, test_rmse=162.4,
    promoted_to_staging=True,
    notes='Added engineered geometry features. Significant improvement. Promote to staging.'
)

print('Experiment Log Comparison')
print('=' * 70)
for run in [run_1, run_2]:
    promoted = 'PROMOTED' if run.promoted_to_staging else 'not promoted'
    print(f'  Run: {run.run_id}  ({promoted})')
    print(f'    Model:    {run.model_class}')
    print(f'    Params:   {run.hyperparameters}')
    print(f'    CV R²:    {run.cv_r2_mean:.3f} ± {run.cv_r2_std:.3f}')
    print(f'    Test R²:  {run.test_r2:.3f}  |  MAE: {run.test_mae:.1f} kN')
    print(f'    Dataset:  {run.dataset_name} (hash: {run.dataset_hash})')
    print(f'    Note:     {run.notes}')
    print()

# Save to JSON for audit trail
log = {'experiments': [asdict(run_1), asdict(run_2)]}
with open('experiment_log.json', 'w') as f:
    json.dump(log, f, indent=2)
print('Experiment log saved to experiment_log.json')

---

## Part 6 — Reproducibility Checklist

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY — Code template with all best practices embedded
# ─────────────────────────────────────────────────────────────────────────

REPRODUCIBILITY_CHECKLIST = """
Reproducibility Checklist for Geotechnical ML Projects
======================================================

CODE
----
[x] np.random.seed(42) set at top of every notebook
[x] random_state=42 passed to every sklearn estimator
[x] torch.manual_seed(42) set if using PyTorch
[x] All shuffle operations use explicit random_state

DATA
----
[x] Dataset version stored in experiment log
[x] SHA-256 hash of CSV stored in experiment log
[x] Raw data never modified; transformations only in code
[x] train/test split indices saved if re-use is needed
[x] Data cleaning steps fully documented (no manual edits in Excel)

ENVIRONMENT
-----------
[x] requirements.txt committed with pinned versions
[x] environment.yml committed for conda users
[x] Python version documented (e.g., 3.11.5)
[x] OS and hardware noted for timing-sensitive benchmarks

MODEL
-----
[x] All hyperparameters logged (not just the best ones)
[x] Model serialised with joblib.dump() or torch.save()
[x] Model card written before deployment
[x] Experiment runs tracked in JSON / MLflow / Weights&Biases

REPOSITORY
----------
[x] .gitignore excludes: *.pyc, __pycache__, .env, *.joblib (use Git LFS for large models)
[x] README.md covers: problem, data, model, results, installation, usage
[x] CHANGELOG.md updated for each model version bump
[x] LICENSE file present
"""

print(REPRODUCIBILITY_CHECKLIST)

# Show what requirements.txt and environment.yml look like
REQUIREMENTS_TXT = """
# requirements.txt — pinned versions for reproducibility
numpy==1.26.4
pandas==2.2.0
scikit-learn==1.4.1
xgboost==2.0.3
matplotlib==3.8.3
seaborn==0.13.2
joblib==1.3.2
jupyter==1.0.0
"""

ENVIRONMENT_YML = """
# environment.yml — conda environment
name: geotech-ml
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.11.5
  - numpy=1.26.4
  - pandas=2.2.0
  - scikit-learn=1.4.1
  - xgboost=2.0.3
  - matplotlib=3.8.3
  - seaborn=0.13.2
  - pip:
    - joblib==1.3.2
"""

print('\nrequirements.txt sample:')
print(REQUIREMENTS_TXT)
print('environment.yml sample:')
print(ENVIRONMENT_YML)

---

## Part 7 — How to Write the README for a Geotechnical ML Repo

A great README answers five questions a visitor has in the first 60 seconds:

1. **What problem does this solve?** (geotechnical context)
2. **What data does it need?** (inputs, format, units)
3. **What model does it use and how well does it work?** (model + metrics)
4. **How do I install and run it?** (commands, no guessing)
5. **Can I trust it?** (limitations, license, citation)

---

### README Template for a Geotechnical ML Repo

```markdown
# Pile Load Capacity Predictor

ML model for predicting driven/bored/CFA pile load test capacity from site
investigation data. Trained on 200 historical pile load tests across sand,
clay, and mixed soil profiles.

## Problem
Static pile load tests cost $50 000–$200 000 each. This model screens pile
configurations early in design to reduce the number of tests required.

## Data
- Input: CSV with 10 features (pile geometry, SPT N values, soil profile)
- Target: Measured pile capacity in kN
- See `data/README.md` for column definitions and units

## Model
- Algorithm: Gradient Boosting Regressor (scikit-learn)
- Features: 15 (10 raw + 5 engineered geometry/resistance features)
- Performance: R² = 0.95 | MAE = 98 kN on held-out test set

## Results
![Feature Importance](figures/feature_importance.png)

## Installation
    git clone https://github.com/your-username/pile-capacity-ml.git
    cd pile-capacity-ml
    pip install -r requirements.txt

## Usage
    python predict.py --input new_sites.csv --output predictions.csv

Or use the Jupyter notebook `notebooks/day_097_end_to_end.ipynb`.

## Limitations
- Do NOT use for final design without independent verification
- Training data covers diameters 0.3–1.2 m and lengths 8–35 m only
- Not validated for offshore piles or karst terrain

## License
MIT License. See LICENSE file.

## Citation
    Malo, R. C. (2026). Pile Load Capacity Predictor v1.0.
    100 Days of ML for Geotechnical Engineering. GitHub.
```

---

## Part 8 — Fully Documented Geotechnical ML Function (End-to-End Demo)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FULLY DOCUMENTED END-TO-END FUNCTION
# Demonstrates: type hints, Google docstring, reproducibility, error handling
# ─────────────────────────────────────────────────────────────────────────
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
import warnings


def train_liquefaction_classifier(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    n_estimators: int = 200,
    max_depth: int = 4,
    learning_rate: float = 0.10,
    cv_folds: int = 5,
    random_state: int = 42,
    verbose: bool = True,
) -> Tuple[Pipeline, Dict]:
    """
    Train a Gradient Boosting Classifier for liquefaction susceptibility.

    The function:
    1. Validates inputs (shape, binary label check)
    2. Builds a StandardScaler → GBClassifier pipeline
    3. Evaluates with stratified k-fold CV (AUC-ROC)
    4. Fits the final model on the full training set
    5. Returns the fitted pipeline and a metrics dictionary

    The caller is responsible for the train/test split and for
    final hold-out evaluation before any deployment decision.

    Args:
        X: Feature DataFrame. Expected columns: depth_m, N60, fines_pct,
           wt_depth_m, PGA_g, Mw, Vs30_mps. All numeric, no NaN.
        y: Binary target Series (1 = liquefied, 0 = not liquefied).
        n_estimators: Number of boosting rounds. Default 200.
        max_depth: Maximum tree depth. Default 4. Range [2, 8] recommended.
        learning_rate: Shrinkage factor. Default 0.10. Range [0.01, 0.30].
        cv_folds: Number of cross-validation folds. Default 5.
        random_state: Global random seed for reproducibility. Default 42.
        verbose: Print CV results if True.

    Returns:
        Tuple of:
          - fitted Pipeline (StandardScaler + GradientBoostingClassifier)
          - metrics dict with keys:
              cv_auc_mean (float), cv_auc_std (float),
              n_train (int), n_features (int), hyperparameters (dict)

    Raises:
        ValueError: If X contains NaN values.
        ValueError: If y is not binary (not exactly 2 unique values).
        ValueError: If X and y have different lengths.

    Example:
        >>> model, metrics = train_liquefaction_classifier(X_train, y_train)
        >>> print(f'AUC = {metrics["cv_auc_mean"]:.3f}')
        AUC = 0.921
        >>> y_proba = model.predict_proba(X_test)[:, 1]
    """
    # ── 1. Input validation ──────────────────────────────────────────────
    if len(X) != len(y):
        raise ValueError(f'X has {len(X)} rows but y has {len(y)} elements.')

    if X.isnull().any().any():
        missing = X.isnull().sum()[X.isnull().sum() > 0].to_dict()
        raise ValueError(f'X contains NaN values: {missing}')

    unique_labels = set(y.unique())
    if unique_labels != {0, 1}:
        raise ValueError(
            f'y must be binary (0/1). Found: {unique_labels}'
        )

    # ── 2. Build pipeline ────────────────────────────────────────────────
    pipe = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            random_state=random_state,
        ))
    ])

    # ── 3. Cross-validation ──────────────────────────────────────────────
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    cv_auc = cross_val_score(pipe, X, y, cv=skf, scoring='roc_auc')

    if verbose:
        print(f'Cross-validation AUC-ROC ({cv_folds}-fold):')
        for fold_i, auc_i in enumerate(cv_auc, 1):
            print(f'  Fold {fold_i}: {auc_i:.4f}')
        print(f'  Mean: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

    # ── 4. Final fit on full training set ────────────────────────────────
    pipe.fit(X, y)

    # ── 5. Return model + metrics ────────────────────────────────────────
    metrics = {
        'cv_auc_mean':    float(cv_auc.mean()),
        'cv_auc_std':     float(cv_auc.std()),
        'n_train':        int(len(X)),
        'n_features':     int(X.shape[1]),
        'hyperparameters': {
            'n_estimators':  n_estimators,
            'max_depth':     max_depth,
            'learning_rate': learning_rate,
            'random_state':  random_state,
        }
    }

    return pipe, metrics


# ─── Demonstrate with synthetic data ─────────────────────────────────────
np.random.seed(42)
N = 200
X_demo = pd.DataFrame({
    'depth_m':    np.random.uniform(1, 15, N),
    'N60':        np.random.randint(4, 30, N).astype(float),
    'fines_pct':  np.random.uniform(0, 35, N),
    'wt_depth_m': np.random.uniform(0.5, 5, N),
    'PGA_g':      np.random.uniform(0.05, 0.60, N),
    'Mw':         np.random.uniform(5.9, 8.0, N),
    'Vs30_mps':   np.random.uniform(100, 400, N),
})
# Simple synthetic label: liquefy if N60 < 15 and PGA > 0.2
y_demo = ((X_demo['N60'] < 15) & (X_demo['PGA_g'] > 0.20)).astype(int)

print('Training liquefaction classifier on synthetic data...')
print()
model, metrics = train_liquefaction_classifier(X_demo, y_demo)
print()
print('Final metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

---

## Key Takeaways

1. **Google-style docstrings are the professional standard.** Args, Returns, Raises, and a runnable Example block are the minimum for any function that another engineer will call. The units of every geotechnical parameter must be stated explicitly.

2. **Type hints make code self-documenting.** `N60: float` removes ambiguity that a comment cannot. Modern IDEs use type hints for autocomplete and static analysis — they are free documentation.

3. **A model card separates intended use from misuse.** Stating what your model cannot do (offshore piles, cemented sands) is as important as stating what it can do — especially when decisions affect structural safety.

4. **Experiment tracking creates an audit trail.** Every change in hyperparameters, feature set, or dataset version should produce a new log entry. SHA-256 hashing of data files pins the exact dataset used in each run.

5. **Reproducibility requires four things in sync:** random seeds, versioned data, pinned requirements, and documented preprocessing. Missing any one makes results irreproducible.

---

*Day 098 of 100 — Undocumented models are technical debt; documented models are engineering assets!*